# checkpoints_train.ipynb — high-quality per-model training (Kaggle H100)


Обучение каждой модели в отдельной ячейке. Все чекпоинты сохраняются в `/kaggle/working/*` для последующего `after_checkpoints.ipynb`.


In [ ]:

# ===== Global config + utils (adapted from utils.py) =====
import os
import gc
import sys
import time
import random
import warnings
from pathlib import Path

import numpy as np
import polars as pl
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

SEED = 1234
N_FOLDS = 5
FORCE_GPU = True
DATA_DIR = "/kaggle/input/data-fusion-contest-2026/"
ROOT_DIR = Path('/kaggle/working')
FEATURES_DIR = ROOT_DIR / 'features'
CKPT_NN = ROOT_DIR / 'checkpoints_nn'
CKPT_LGBM = ROOT_DIR / 'checkpoints_lgbm'
CKPT_PB = ROOT_DIR / 'checkpoints_pyboost'
CKPT_LGBM_META = ROOT_DIR / 'checkpoints_lgbm_meta'
BLEND_ART = ROOT_DIR / 'blend_artifacts'
SUB_DIR = ROOT_DIR / 'submissions'
for p in [FEATURES_DIR, CKPT_NN, CKPT_LGBM, CKPT_PB, CKPT_LGBM_META, BLEND_ART, SUB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def get_device():
    import torch
    if torch.cuda.is_available():
        return torch.device('cuda')
    raise RuntimeError('GPU is required: CUDA device is not available in this Kaggle session')

def compute_macro_auc(y_true, y_pred, target_cols):
    aucs = {}
    for i, col in enumerate(target_cols):
        y_t = y_true[:, i]
        if y_t.sum() >= 2 and (len(y_t) - y_t.sum()) >= 2:
            aucs[col] = roc_auc_score(y_t, y_pred[:, i])
    return float(np.mean(list(aucs.values()))), aucs

def to_ranks(arr):
    return np.column_stack([rankdata(arr[:, i]) for i in range(arr.shape[1])]).astype(np.float64)

def verify_submission(submit, sample):
    assert submit.shape == sample.shape, f"Shape mismatch: {submit.shape} vs {sample.shape}"
    assert submit.columns == sample.columns, 'Column mismatch'
    for col in submit.columns:
        assert submit[col].dtype == sample[col].dtype, f"Dtype mismatch for {col}"
    print(f"Format verified: {submit.shape[0]:,} rows, {submit.shape[1]} cols")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

seed_everything(SEED)
print('DATA_DIR =', DATA_DIR)


## 1) Feature engineering code


In [ ]:
"""Step 1: Feature Engineering.

Loads raw parquet data, builds all features, saves to features/ directory.
Subsequent scripts load from features/ instead of raw data.

Runtime: ~3-5 minutes.
"""

import gc
import hashlib
import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from scipy.sparse import csr_matrix, hstack as sparse_hstack
from sklearn.decomposition import TruncatedSVD



FEATURES_DIR = FEATURES_DIR
NULL_PCA_COMPONENTS = 20
INDIVIDUAL_NULL_THRESHOLD = 0.05

# ── Feature Engineering Config ────────────────────────────────────

NULL_GROUPS_MAIN = {
    "null_group_A": "num_feature_1",
    "null_group_B": "num_feature_8",
    "null_group_C": "num_feature_11",
    "null_group_D": "num_feature_38",
    "null_group_E": "num_feature_72",
}

NULL_GROUPS_EXTRA = {
    "null_extra_E4": "num_feature_155",
    "null_extra_E5": "num_feature_140",
    "null_extra_E8": "num_feature_139",
    "null_extra_E9": "num_feature_222",
    "null_extra_E10": "num_feature_134",
    "null_extra_E11": "num_feature_158",
    "null_extra_E12": "num_feature_240",
    "null_extra_E13": "num_feature_194",
    "null_extra_E14": "num_feature_197",
    "null_extra_E15": "num_feature_167",
}

NUM_DIFFS = [
    ("num_feature_7", "num_feature_42"),
    ("num_feature_33", "num_feature_42"),
    ("num_feature_36", "num_feature_42"),
    ("num_feature_7", "num_feature_132"),
    ("num_feature_33", "num_feature_29"),
    ("num_feature_7", "num_feature_57"),
    ("num_feature_33", "num_feature_132"),
    ("num_feature_7", "num_feature_125"),
    ("num_feature_29", "num_feature_42"),
    ("num_feature_41", "num_feature_42"),
    ("num_feature_56", "num_feature_34"),
    ("num_feature_7", "num_feature_34"),
    ("num_feature_33", "num_feature_13"),
    ("num_feature_7", "num_feature_118"),
    ("num_feature_36", "num_feature_34"),
    ("num_feature_29", "num_feature_13"),
    ("num_feature_118", "num_feature_42"),
    ("num_feature_125", "num_feature_118"),
]

CAT_INTERACTIONS = [
    ("cat_feature_66", "cat_feature_46"),
    ("cat_feature_66", "cat_feature_39"),
    ("cat_feature_66", "cat_feature_48"),
    ("cat_feature_66", "cat_feature_9"),
    ("cat_feature_66", "cat_feature_52"),
]

RATIO_FEATURES = [
    ("num_feature_62", "num_feature_79"),
    ("num_feature_27", "num_feature_79"),
    ("num_feature_76", "num_feature_79"),
    ("num_feature_62", "num_feature_86"),
    ("num_feature_116", "num_feature_124"),
    ("num_feature_36", "num_feature_79"),
    ("num_feature_41", "num_feature_79"),
    ("num_feature_83", "num_feature_108"),
    ("num_feature_83", "num_feature_103"),
]

DUPLICATE_CATS = [
    "cat_feature_24",
    "cat_feature_25",
    "cat_feature_26",
    "cat_feature_29",
    "cat_feature_50",
    "cat_feature_63",
]


# ── Helper functions ──────────────────────────────────────────────


def filter_extra_features(train_extra, test_extra):
    """Remove extra features that are >99% null or constant."""
    extra_cols = [c for c in train_extra.columns if c != "customer_id"]
    n_rows = train_extra.height
    to_drop = set()
    for col in extra_cols:
        s = train_extra[col]
        if s.null_count() / n_rows > 0.99:
            to_drop.add(col)
            continue
        non_null = s.drop_nulls()
        if len(non_null) == 0 or non_null.n_unique() <= 1:
            to_drop.add(col)
    keep_cols = ["customer_id"] + [c for c in extra_cols if c not in to_drop]
    print(f"  Extra: {len(extra_cols)} total, {len(to_drop)} removed, "
          f"{len(keep_cols) - 1} kept")
    return train_extra.select(keep_cols), test_extra.select(keep_cols)


def deduplicate_extra_features(train_extra, test_extra):
    """Remove exact duplicate columns using MD5 hashing."""
    extra_cols = [c for c in train_extra.columns if c.startswith("num_feature")]
    seen_hashes = {}
    dup_cols = []
    BATCH = 200
    for start in range(0, len(extra_cols), BATCH):
        batch_cols = extra_cols[start:start + BATCH]
        batch_np = train_extra.select(batch_cols).to_numpy()
        for i, col in enumerate(batch_cols):
            h = hashlib.md5(batch_np[:, i].tobytes()).hexdigest()
            if h in seen_hashes:
                dup_cols.append(col)
            else:
                seen_hashes[h] = col
        del batch_np
    if dup_cols:
        train_extra = train_extra.drop(dup_cols)
        test_extra = test_extra.drop(dup_cols)
        print(f"  Dedup: removed {len(dup_cols)} → {len(extra_cols) - len(dup_cols)} unique")
    return train_extra, test_extra


def null_pattern_pca(train_extra_raw, test_extra_raw, n_components=20):
    """PCA of binary null-indicator matrix from all extra columns (sparse)."""
    all_cols = [c for c in train_extra_raw.columns if c.startswith("num_feature")]
    n_train = train_extra_raw.height
    BATCH = 500
    sparse_blocks = []
    for start in range(0, len(all_cols), BATCH):
        batch_cols = all_cols[start:start + BATCH]
        train_np = train_extra_raw.select(batch_cols).to_numpy()
        test_np = test_extra_raw.select(batch_cols).to_numpy()
        combined = np.vstack([train_np, test_np])
        del train_np, test_np
        null_bits = np.isnan(combined).astype(np.float32)
        sparse_blocks.append(csr_matrix(null_bits))
        del combined, null_bits
    null_sparse = sparse_hstack(sparse_blocks, format="csr")
    del sparse_blocks; gc.collect()

    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    pca_features = svd.fit_transform(null_sparse)
    var_explained = svd.explained_variance_ratio_.sum()
    print(f"  Null PCA: {null_sparse.shape[1]} cols → {n_components} components, "
          f"var explained: {var_explained:.3f}")
    del null_sparse; gc.collect()

    return pca_features[:n_train].astype(np.float32), pca_features[n_train:].astype(np.float32)


def add_null_indicators(df, groups_dict):
    """Add binary null-indicator features from named groups."""
    new_cols = []
    for name, ref_col in groups_dict.items():
        if ref_col in df.columns:
            new_cols.append(pl.col(ref_col).is_null().cast(pl.Int8).alias(name))
    if new_cols:
        df = df.with_columns(new_cols)
    return df


def select_null_indicator_features(train_main, num_cols, train_tgt, target_cols, threshold):
    """Select num features whose null indicator has |corr| > threshold with any target."""
    null_arrays, valid_cols = [], []
    for col in num_cols:
        arr = train_main[col].is_null().cast(pl.Float32).to_numpy()
        if arr.std() < 1e-8:
            continue
        null_arrays.append(arr)
        valid_cols.append(col)
    if not null_arrays:
        return []

    null_matrix = np.column_stack(null_arrays).astype(np.float32)
    tgt_np = train_tgt.select(target_cols).to_numpy().astype(np.float32)

    null_mean = null_matrix.mean(axis=0, keepdims=True)
    null_std = null_matrix.std(axis=0, keepdims=True)
    null_std[null_std < 1e-8] = 1.0
    null_matrix = (null_matrix - null_mean) / null_std

    tgt_mean = tgt_np.mean(axis=0, keepdims=True)
    tgt_std = tgt_np.std(axis=0, keepdims=True)
    tgt_std[tgt_std < 1e-8] = 1.0
    tgt_np = (tgt_np - tgt_mean) / tgt_std

    corr_matrix = (null_matrix.T @ tgt_np) / null_matrix.shape[0]
    max_abs_corr = np.abs(corr_matrix).max(axis=1)
    return [col for col, c in zip(valid_cols, max_abs_corr) if c > threshold]


def add_individual_null_indicators(df, selected_cols):
    """Add is_null indicators for selected numerical features."""
    exprs = [pl.col(c).is_null().cast(pl.Int8).alias(f"is_null_{c}") for c in selected_cols]
    return df.with_columns(exprs)


def add_frequency_encoding(train_df, test_df, cat_cols):
    """Frequency-encode categoricals using train+test combined counts."""
    total = train_df.height + test_df.height
    freq_cols = []
    for col in cat_cols:
        fname = f"freq_{col}"
        combined = pl.concat([train_df.select(col), test_df.select(col)])
        freq_map = (
            combined[col].value_counts()
            .with_columns((pl.col("count") / total).alias(fname))
            .select([col, fname])
        )
        train_df = train_df.join(freq_map, on=col, how="left")
        test_df = test_df.join(freq_map, on=col, how="left")
        freq_cols.append(fname)
    return train_df, test_df, freq_cols


def add_cat_interaction_freqs(train_df, test_df, interactions):
    """Add frequency-encoded category interaction features."""
    total = train_df.height + test_df.height
    freq_cols = []
    for c1, c2 in interactions:
        fname = f"freq_{c1}_x_{c2}"
        combo = pl.col(c1).cast(pl.Int64) * 1_000_000 + pl.col(c2).cast(pl.Int64)
        combined = pl.concat([
            train_df.select(combo.alias("_combo")),
            test_df.select(combo.alias("_combo")),
        ])
        freq_map = (
            combined["_combo"].value_counts()
            .with_columns((pl.col("count") / total).alias(fname))
            .select(["_combo", fname])
        )
        train_df = (train_df.with_columns(combo.alias("_combo"))
                    .join(freq_map, on="_combo", how="left").drop("_combo"))
        test_df = (test_df.with_columns(combo.alias("_combo"))
                   .join(freq_map, on="_combo", how="left").drop("_combo"))
        freq_cols.append(fname)
    return train_df, test_df, freq_cols


def add_numerical_diffs(df, diff_pairs):
    """Add difference features for numerical pairs."""
    exprs = []
    for a, b in diff_pairs:
        if a in df.columns and b in df.columns:
            a_id = a.replace("num_feature_", "")
            b_id = b.replace("num_feature_", "")
            exprs.append((pl.col(a) - pl.col(b)).alias(f"diff_{a_id}_minus_{b_id}"))
    if exprs:
        df = df.with_columns(exprs)
    return df


def add_ratio_features(df, ratio_pairs):
    """Add ratio features a/b."""
    exprs = []
    for a, b in ratio_pairs:
        name = f"ratio_{a.replace('num_feature_', '')}_{b.replace('num_feature_', '')}"
        exprs.append(
            pl.when(pl.col(b).abs() > 1e-8)
            .then(pl.col(a) / pl.col(b))
            .otherwise(None)
            .alias(name)
        )
    return df.with_columns(exprs)


def add_null_count(df, cols, name, batch_size=300):
    """Add per-row null count for given columns."""
    parts = []
    for i in range(0, len(cols), batch_size):
        batch = cols[i:i + batch_size]
        parts.append(pl.sum_horizontal([pl.col(c).is_null().cast(pl.UInt16) for c in batch]))
    total = parts[0]
    for p in parts[1:]:
        total = total + p
    return df.with_columns(total.alias(name))


def add_row_mean(df, num_cols):
    """Add per-row mean of numerical features."""
    return df.with_columns(
        pl.mean_horizontal([pl.col(c) for c in num_cols]).alias("row_mean_main")
    )


def add_row_stats(df, num_cols, prefix):
    """Add row-level std and skew for numerical columns."""
    arr = df.select(num_cols).to_numpy()
    row_mean = np.nanmean(arr, axis=1, keepdims=True)
    diff = arr - row_mean
    m2 = np.nanmean(diff ** 2, axis=1)
    m3 = np.nanmean(diff ** 3, axis=1)
    row_std = np.sqrt(np.maximum(m2, 0)).astype(np.float32)
    with np.errstate(divide="ignore", invalid="ignore"):
        row_skew = np.where(m2 > 1e-16, m3 / (m2 ** 1.5 + 1e-16), 0).astype(np.float32)
    del arr, diff, m2, m3; gc.collect()
    return df.with_columns([
        pl.Series(f"{prefix}_row_std", row_std),
        pl.Series(f"{prefix}_row_skew", row_skew),
    ])


def remove_duplicate_cats(train_df, test_df, dup_cols):
    """Remove duplicate categorical features."""
    existing = [c for c in dup_cols if c in train_df.columns]
    if existing:
        train_df = train_df.drop(existing)
        test_df = test_df.drop(existing)
        print(f"  Removed {len(existing)} duplicate cats")
    return train_df, test_df


# ── Main pipeline ─────────────────────────────────────────────────


def main_01():
    t0 = time.time()
    print("=" * 60)
    print("Step 1: Feature Engineering")
    print("=" * 60)

    # 1. Load raw data
    print("\n[1/8] Loading raw data...")
    train_main = pl.read_parquet(f"{DATA_DIR}train_main_features.parquet")
    test_main = pl.read_parquet(f"{DATA_DIR}test_main_features.parquet")
    train_extra_raw = pl.read_parquet(f"{DATA_DIR}train_extra_features.parquet")
    test_extra_raw = pl.read_parquet(f"{DATA_DIR}test_extra_features.parquet")
    train_tgt = pl.read_parquet(f"{DATA_DIR}train_target.parquet")
    print(f"  Train: main {train_main.shape}, extra {train_extra_raw.shape}")

    cat_cols = sorted([c for c in train_main.columns if c.startswith("cat_feature")])
    num_cols_main = sorted([c for c in train_main.columns if c.startswith("num_feature")])
    target_cols = [c for c in train_tgt.columns if c.startswith("target_")]

    # 2. Null pattern PCA (on raw extra, before filtering)
    print("\n[2/8] Null Pattern PCA...")
    train_null_pca, test_null_pca = null_pattern_pca(
        train_extra_raw, test_extra_raw, NULL_PCA_COMPONENTS
    )

    # 3. Filter + dedup extra
    print("\n[3/8] Filter + dedup extra features...")
    train_extra, test_extra = filter_extra_features(train_extra_raw, test_extra_raw)
    del train_extra_raw, test_extra_raw; gc.collect()
    train_extra, test_extra = deduplicate_extra_features(train_extra, test_extra)

    # 4. Remove duplicate cats + cast
    print("\n[4/8] Prepare categoricals...")
    train_main, test_main = remove_duplicate_cats(train_main, test_main, DUPLICATE_CATS)
    cat_cols = sorted([c for c in train_main.columns if c.startswith("cat_feature")])
    train_main = train_main.with_columns(pl.col(cat_cols).cast(pl.Int32))
    test_main = test_main.with_columns(pl.col(cat_cols).cast(pl.Int32))

    # 5. Feature engineering
    print("\n[5/8] Engineering features...")

    # Null indicators (groups)
    train_main = add_null_indicators(train_main, NULL_GROUPS_MAIN)
    test_main = add_null_indicators(test_main, NULL_GROUPS_MAIN)
    train_extra = add_null_indicators(train_extra, NULL_GROUPS_EXTRA)
    test_extra = add_null_indicators(test_extra, NULL_GROUPS_EXTRA)
    print(f"  + {len(NULL_GROUPS_MAIN) + len(NULL_GROUPS_EXTRA)} null group indicators")

    # Null counts
    train_main = add_null_count(train_main, num_cols_main, "null_count_main")
    test_main = add_null_count(test_main, num_cols_main, "null_count_main")
    extra_num_cols = [c for c in train_extra.columns if c.startswith("num_feature")]
    train_extra = add_null_count(train_extra, extra_num_cols, "null_count_extra")
    test_extra = add_null_count(test_extra, extra_num_cols, "null_count_extra")
    print(f"  + 2 null count features")

    # Frequency encoding
    train_main, test_main, freq_cols = add_frequency_encoding(train_main, test_main, cat_cols)
    print(f"  + {len(freq_cols)} frequency features")

    # Category interaction frequencies
    train_main, test_main, interact_cols = add_cat_interaction_freqs(
        train_main, test_main, CAT_INTERACTIONS
    )
    print(f"  + {len(interact_cols)} interaction features")

    # Numerical diffs
    train_main = add_numerical_diffs(train_main, NUM_DIFFS)
    test_main = add_numerical_diffs(test_main, NUM_DIFFS)
    diff_cols = [c for c in train_main.columns if c.startswith("diff_")]
    print(f"  + {len(diff_cols)} diff features")

    # Ratio features
    train_main = add_ratio_features(train_main, RATIO_FEATURES)
    test_main = add_ratio_features(test_main, RATIO_FEATURES)
    print(f"  + {len(RATIO_FEATURES)} ratio features")

    # Row mean
    train_main = add_row_mean(train_main, num_cols_main)
    test_main = add_row_mean(test_main, num_cols_main)
    print(f"  + 1 row_mean feature")

    # Individual null indicators (correlation-selected)
    null_ind_cols = select_null_indicator_features(
        train_main, num_cols_main, train_tgt, target_cols, INDIVIDUAL_NULL_THRESHOLD
    )
    train_main = add_individual_null_indicators(train_main, null_ind_cols)
    test_main = add_individual_null_indicators(test_main, null_ind_cols)
    print(f"  + {len(null_ind_cols)} individual null indicators (|corr| > {INDIVIDUAL_NULL_THRESHOLD})")

    # Row stats
    train_main = add_row_stats(train_main, num_cols_main, "main")
    test_main = add_row_stats(test_main, num_cols_main, "main")
    extra_num_for_stats = [c for c in train_extra.columns if c.startswith("num_feature")]
    train_extra = add_row_stats(train_extra, extra_num_for_stats, "extra")
    test_extra = add_row_stats(test_extra, extra_num_for_stats, "extra")
    print(f"  + 4 row stats (main + extra std/skew)")

    # 6. Join main + extra + null PCA
    print("\n[6/8] Joining features...")
    train_feat = train_main.join(train_extra, on="customer_id")
    test_feat = test_main.join(test_extra, on="customer_id")
    del train_main, train_extra, test_main, test_extra; gc.collect()

    null_pca_cols = [f"null_pca_{i}" for i in range(NULL_PCA_COMPONENTS)]
    train_feat = train_feat.hstack(pl.DataFrame(train_null_pca, schema=null_pca_cols))
    test_feat = test_feat.hstack(pl.DataFrame(test_null_pca, schema=null_pca_cols))
    del train_null_pca, test_null_pca

    feature_cols = [c for c in train_feat.columns if c != "customer_id"]
    cat_feature_names = [c for c in feature_cols if c.startswith("cat_feature")]
    num_feature_names = [c for c in feature_cols if c not in cat_feature_names]

    print(f"  Total: {len(feature_cols)} features "
          f"({len(cat_feature_names)} cat, {len(num_feature_names)} num)")

    # 7. Save targets
    print("\n[7/8] Saving targets...")
    FEATURES_DIR.mkdir(exist_ok=True)
    targets = train_tgt.select(["customer_id"] + target_cols)
    targets.write_parquet(FEATURES_DIR / "targets.parquet")

    # 8. Save features + metadata
    print("\n[8/8] Saving features...")
    train_feat.write_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat.write_parquet(FEATURES_DIR / "test_features.parquet")

    meta = {
        "cat_cols": cat_feature_names,
        "num_cols": num_feature_names,
        "feature_names": feature_cols,
        "target_cols": target_cols,
    }
    with open(FEATURES_DIR / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    elapsed = time.time() - t0
    print(f"\nDone in {elapsed / 60:.1f} min.")
    print(f"  Saved to {FEATURES_DIR}/")
    print(f"  Features: {len(feature_cols)}")
    print(f"  Train: {train_feat.shape}, Test: {test_feat.shape}")




In [ ]:
# Run Feature Engineering
seed_everything(SEED)
main_01()


## 2) NN training code (DAE + TabM)


In [ ]:
"""Step 2: Train Neural Network (TabM + PLR + ASL + Mixup + SWA).

Loads features from features/ directory, trains DAE for semi-supervised
bottleneck embeddings, then trains TabM with PLR encoding.

Output: checkpoints_nn/fold_{0-4}.npz (val_preds, test_preds, val_idx)

Runtime: ~2-3 hours on GPU, ~5-8 hours on MPS/CPU.
"""

import gc
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import QuantileTransformer
from torch.utils.data import DataLoader, TensorDataset
DEVICE = get_device()
FEATURES_DIR = FEATURES_DIR
CHECKPOINT_DIR = CKPT_NN

# ── Hyperparameters ───────────────────────────────────────────────
BATCH_SIZE = 1024
EPOCHS = 50
LR = 5e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 10
GRAD_CLIP = 1.0
HIDDEN_DIM = 384

# DAE
DAE_BOTTLENECK_DIM = 256
DAE_EPOCHS = 20
DAE_LR = 1e-3
DAE_BATCH_SIZE = 4096
DAE_CORRUPTION_RATE = 0.15
DAE_WEIGHT_DECAY = 1e-5
DAE_GRAD_CLIP = 1.0

# TabM + ASL + Mixup + PLR
K_ENSEMBLE = 16
ASL_GAMMA_NEG = 4
ASL_GAMMA_POS = 1
ASL_CLIP = 0.05
MIXUP_ALPHA = 0.2
PLR_N_BINS = 24
TOP_K_SWA = 5


# ══════════════════════════════════════════════════════════════════
#  Denoising Autoencoder
# ══════════════════════════════════════════════════════════════════


class DenoisingAutoencoder(nn.Module):
    """Symmetric DAE: Input → 1024 → 512 → 256 (bottleneck) → 512 → 1024 → Input."""

    def __init__(self, input_dim, bottleneck_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024), nn.BatchNorm1d(1024), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, bottleneck_dim), nn.BatchNorm1d(bottleneck_dim), nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 512), nn.BatchNorm1d(512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(1024, input_dim),
        )
        for module in [self.encoder, self.decoder]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                    nn.init.zeros_(layer.bias)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def apply_swap_noise(clean, non_null_mask, corruption_rate):
    """Apply swap noise: replace values with random other rows in the batch."""
    corrupt_mask = (torch.rand_like(clean) < corruption_rate) & non_null_mask
    perm = torch.randperm(clean.shape[0], device=clean.device)
    return torch.where(corrupt_mask, clean[perm], clean)


def train_dae(all_num_data, null_mask_np, device):
    """Train DAE on train+test numerical data (semi-supervised)."""
    n_samples, input_dim = all_num_data.shape
    print(f"    DAE: {n_samples:,} × {input_dim}, bottleneck={DAE_BOTTLENECK_DIM}")

    model = DenoisingAutoencoder(input_dim, DAE_BOTTLENECK_DIM).to(device)
    use_compile = hasattr(torch, "compile")
    if use_compile:
        backend = "aot_eager" if device.type == "mps" else "inductor"
        model = torch.compile(model, backend=backend)

    optimizer = torch.optim.AdamW(model.parameters(), lr=DAE_LR, weight_decay=DAE_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=DAE_EPOCHS)

    data_tensor = torch.from_numpy(all_num_data).to(device)
    non_null_tensor = torch.from_numpy((1 - null_mask_np).astype(np.float32)).to(device)
    del null_mask_np; gc.collect()

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    n_batches = (n_samples + DAE_BATCH_SIZE - 1) // DAE_BATCH_SIZE

    # Check for saved checkpoint
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    ckpt_final = CHECKPOINT_DIR / "dae_final.pt"
    if ckpt_final.exists():
        print(f"    Loading DAE from {ckpt_final}")
        ckpt = torch.load(ckpt_final, map_location=device, weights_only=True)
        orig_model = model._orig_mod if use_compile else model
        orig_model.load_state_dict(ckpt["model_state_dict"])
        del data_tensor, non_null_tensor, ckpt
        return model

    for epoch in range(DAE_EPOCHS):
        model.train()
        total_loss = 0.0
        perm_indices = torch.randperm(n_samples, device=device)
        for b in range(n_batches):
            start = b * DAE_BATCH_SIZE
            end = min(start + DAE_BATCH_SIZE, n_samples)
            idx = perm_indices[start:end]
            clean = data_tensor[idx]
            non_null = non_null_tensor[idx]

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                corrupted = apply_swap_noise(clean, non_null > 0.5, DAE_CORRUPTION_RATE)
                reconstructed = model(corrupted)
                diff_sq = (reconstructed.float() - clean.float()) ** 2 * non_null
                loss = diff_sq.sum() / non_null.sum().clamp(min=1)

            optimizer.zero_grad()
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), DAE_GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), DAE_GRAD_CLIP)
                optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"      Epoch {epoch+1:2d}/{DAE_EPOCHS}  loss={total_loss/n_batches:.6f}",
                  flush=True)

    orig_model = model._orig_mod if use_compile else model
    torch.save({"epoch": DAE_EPOCHS, "model_state_dict": orig_model.state_dict(),
                "loss": total_loss / n_batches}, ckpt_final)
    del data_tensor, non_null_tensor
    return model


@torch.no_grad()
def extract_dae_embeddings(model, all_num_data, device, batch_size=4096):
    """Extract bottleneck embeddings from trained DAE."""
    model.eval()
    embeddings = []
    for start in range(0, all_num_data.shape[0], batch_size):
        end = min(start + batch_size, all_num_data.shape[0])
        batch = torch.from_numpy(all_num_data[start:end]).to(device)
        embeddings.append(model.encode(batch).cpu().numpy())
    return np.vstack(embeddings)


# ══════════════════════════════════════════════════════════════════
#  TabM Model
# ══════════════════════════════════════════════════════════════════


class LinearBE(nn.Module):
    """Batch Ensemble Linear: shared W + per-member rank-1 adapters."""
    def __init__(self, in_dim, out_dim, k):
        super().__init__()
        self.k = k
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        self.r = nn.Parameter(torch.ones(k, in_dim))
        self.s = nn.Parameter(torch.ones(k, out_dim))
        self.b = nn.Parameter(torch.zeros(k, out_dim))

    def forward(self, x):
        x = x * self.r.unsqueeze(0)
        B, k, D = x.shape
        x = self.linear(x.reshape(B * k, D)).reshape(B, k, -1)
        return x * self.s.unsqueeze(0) + self.b.unsqueeze(0)


class ResidualBlockBE(nn.Module):
    """Batch Ensemble residual block with zero-init."""
    def __init__(self, dim, k, dropout=0.3):
        super().__init__()
        self.bn1 = nn.BatchNorm1d(dim)
        self.lin1 = LinearBE(dim, dim, k)
        self.bn2 = nn.BatchNorm1d(dim)
        self.lin2 = LinearBE(dim, dim, k)
        self.drop = nn.Dropout(dropout)
        self.act = nn.SiLU()
        nn.init.zeros_(self.lin2.linear.weight)

    def forward(self, x):
        B, k, D = x.shape
        h = self.bn1(x.reshape(B * k, D)).reshape(B, k, D)
        h = self.act(h)
        h = self.lin1(h)
        B, k, D = h.shape
        h = self.bn2(h.reshape(B * k, D)).reshape(B, k, D)
        h = self.act(h)
        h = self.drop(h)
        h = self.lin2(h)
        return x + h


class AsymmetricLoss(nn.Module):
    """ASL: down-weights easy negatives via probability shifting."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos_term = (1 - p).pow(self.gamma_pos) * torch.log(p.clamp(min=self.eps))
        p_m = (p - self.clip).clamp(min=self.eps) if self.clip > 0 else p
        neg_term = p_m.pow(self.gamma_neg) * torch.log((1 - p_m).clamp(min=self.eps))
        return (-(targets * pos_term) - (1 - targets) * neg_term).mean()


class PiecewiseLinearEncoding(nn.Module):
    """PLR: learnable piecewise-linear function per feature (d_embedding=1)."""
    def __init__(self, n_features, n_bins=24):
        super().__init__()
        self.n_features = n_features
        self.n_bins = n_bins
        self.register_buffer('edges', torch.zeros(n_features, n_bins + 1))
        self.weight = nn.Parameter(torch.empty(n_features, n_bins))
        self.bias = nn.Parameter(torch.zeros(n_features))
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)

    def set_bins(self, X_train_np):
        quantiles = np.linspace(0, 1, self.n_bins + 1)
        for i in range(self.n_features):
            col = X_train_np[:, i]
            col_valid = col[~np.isnan(col)] if np.any(np.isnan(col)) else col
            if len(col_valid) < 2:
                edges = np.linspace(-3.0, 3.0, self.n_bins + 1)
            else:
                edges = np.unique(np.quantile(col_valid, quantiles))
                if len(edges) < self.n_bins + 1:
                    edges = np.linspace(edges[0], edges[-1], self.n_bins + 1)
                else:
                    edges = edges[:self.n_bins + 1]
            self.edges[i] = torch.from_numpy(edges.astype(np.float32))

    def forward(self, x):
        left = self.edges[:, :-1]
        right = self.edges[:, 1:]
        width = (right - left).clamp(min=1e-8)
        ple = ((x.unsqueeze(2) - left) / width).clamp(0, 1)
        return (ple * self.weight).sum(dim=2) + self.bias


class TabularNet(nn.Module):
    """TabM: Batch Ensemble MLP with PLR + categorical embeddings."""
    def __init__(self, cat_cardinalities, n_numerical, n_targets, k=K_ENSEMBLE):
        super().__init__()
        self.k = k
        self.n_targets = n_targets
        self.plr = PiecewiseLinearEncoding(n_numerical, PLR_N_BINS)

        self.embeddings = nn.ModuleList()
        total_embed_dim = 0
        for n_cat in cat_cardinalities:
            dim = min(50, max(2, (n_cat + 1) // 2))
            self.embeddings.append(nn.Embedding(n_cat + 2, dim))
            total_embed_dim += dim

        input_dim = total_embed_dim + n_numerical
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM), nn.BatchNorm1d(HIDDEN_DIM), nn.SiLU(),
        )
        self.res_blocks = nn.ModuleList([
            ResidualBlockBE(HIDDEN_DIM, k, dropout=0.4),
            ResidualBlockBE(HIDDEN_DIM, k, dropout=0.4),
            ResidualBlockBE(HIDDEN_DIM, k, dropout=0.3),
        ])
        head_dim = HIDDEN_DIM // 2
        self.head_bn1 = nn.BatchNorm1d(HIDDEN_DIM)
        self.head_drop1 = nn.Dropout(0.2)
        self.head_lin1 = LinearBE(HIDDEN_DIM, head_dim, k)
        self.head_bn2 = nn.BatchNorm1d(head_dim)
        self.head_drop2 = nn.Dropout(0.1)
        self.head_lin2 = LinearBE(head_dim, n_targets, k)
        self.act = nn.SiLU()

        for layer in self.input_proj:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)
        nn.init.kaiming_normal_(self.head_lin1.linear.weight, nonlinearity="relu")
        nn.init.kaiming_normal_(self.head_lin2.linear.weight, nonlinearity="relu")

    def _embed_and_project(self, x_cat, x_num):
        x_num = self.plr(x_num)
        embeds = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        return self.input_proj(torch.cat(embeds + [x_num], dim=1))

    def _head(self, x):
        B, k, D = x.shape
        x = self.head_bn1(x.reshape(B * k, D)).reshape(B, k, D)
        x = self.act(x); x = self.head_drop1(x); x = self.head_lin1(x)
        B, k, D2 = x.shape
        x = self.head_bn2(x.reshape(B * k, D2)).reshape(B, k, D2)
        x = self.act(x); x = self.head_drop2(x)
        return self.head_lin2(x)

    def forward(self, x_cat, x_num):
        x = self._embed_and_project(x_cat, x_num).unsqueeze(1).expand(-1, self.k, -1)
        for block in self.res_blocks:
            x = block(x)
        logits = self._head(x)
        if self.training:
            return logits
        return torch.sigmoid(logits).mean(dim=1)

    def forward_mixup(self, x_cat, x_num, lam, perm, mixup_layer):
        x = self._embed_and_project(x_cat, x_num).unsqueeze(1).expand(-1, self.k, -1)
        if mixup_layer == 0:
            x = lam * x + (1 - lam) * x[perm]
        for i, block in enumerate(self.res_blocks):
            x = block(x)
            if mixup_layer == i + 1:
                x = lam * x + (1 - lam) * x[perm]
        return self._head(x)


# ══════════════════════════════════════════════════════════════════
#  Data helpers
# ══════════════════════════════════════════════════════════════════


def to_tensors(df_feat, cat_cols, num_cols, cat_cardinalities, df_tgt=None, target_cols=None):
    """Convert Polars DataFrame to PyTorch tensors."""
    cat_arr = df_feat.select(cat_cols).to_numpy()
    cat_np = np.zeros_like(cat_arr, dtype=np.int64)
    for i, card in enumerate(cat_cardinalities):
        col = cat_arr[:, i]
        null_mask = np.isnan(col) if np.issubdtype(col.dtype, np.floating) else np.zeros(len(col), dtype=bool)
        values = np.clip(np.nan_to_num(col, nan=0).astype(np.int64), 0, card)
        values[null_mask] = card + 1
        cat_np[:, i] = values

    num_arr = df_feat.select(num_cols).fill_null(0.0).to_numpy().astype(np.float32)
    num_arr = np.nan_to_num(num_arr, nan=0.0, posinf=0.0, neginf=0.0)

    if df_tgt is not None and target_cols is not None:
        tgt_arr = df_tgt.select(target_cols).to_numpy().astype(np.float32)
        return torch.from_numpy(cat_np), torch.from_numpy(num_arr), torch.from_numpy(tgt_arr)
    return torch.from_numpy(cat_np), torch.from_numpy(num_arr)


def quantile_normalize(train_num, val_num, test_num):
    """Quantile-transform to N(0,1). Fit on train only."""
    n = train_num.shape[0]
    qt = QuantileTransformer(n_quantiles=min(1000, n), output_distribution="normal",
                             subsample=min(100_000, n), random_state=SEED)
    tr_np = train_num.numpy()
    qt.fit(tr_np)
    return (
        torch.from_numpy(np.nan_to_num(qt.transform(tr_np), nan=0.0).astype(np.float32)),
        torch.from_numpy(np.nan_to_num(qt.transform(val_num.numpy()), nan=0.0).astype(np.float32)),
        torch.from_numpy(np.nan_to_num(qt.transform(test_num.numpy()), nan=0.0).astype(np.float32)),
    )


# ══════════════════════════════════════════════════════════════════
#  Training
# ══════════════════════════════════════════════════════════════════


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n_batches = 0, 0
    k = model.k
    for x_cat, x_num, y in loader:
        x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
        optimizer.zero_grad()
        mixup_layer = np.random.randint(0, 4)
        lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
        perm = torch.randperm(x_cat.shape[0], device=device)
        logits = model.forward_mixup(x_cat, x_num, lam, perm, mixup_layer)
        y_k = y.unsqueeze(1).expand(-1, k, -1)
        y_mixed = lam * y_k + (1 - lam) * y[perm].unsqueeze(1).expand(-1, k, -1)
        loss = criterion(logits, y_mixed)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    for x_cat, x_num, y in loader:
        probs = model(x_cat.to(device), x_num.to(device))
        all_preds.append(probs.cpu())
        all_targets.append(y)
    return torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy()


@torch.no_grad()
def predict_model(model, loader, device):
    model.eval()
    all_preds = []
    for batch in loader:
        probs = model(batch[0].to(device), batch[1].to(device))
        all_preds.append(probs.cpu())
    return torch.cat(all_preds).numpy()


def train_one_fold(fold_idx, tr_cat, tr_num, tr_y, val_cat, val_num, val_y,
                   test_cat, test_num, cat_cardinalities, n_numerical,
                   target_cols, device):
    train_loader = DataLoader(TensorDataset(tr_cat, tr_num, tr_y),
                              batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(TensorDataset(val_cat, val_num, val_y),
                            batch_size=BATCH_SIZE * 2, shuffle=False)
    test_loader = DataLoader(TensorDataset(test_cat, test_num),
                             batch_size=BATCH_SIZE * 2, shuffle=False)

    model = TabularNet(cat_cardinalities, n_numerical, len(target_cols)).to(device)
    if fold_idx == 0:
        n_params = sum(p.numel() for p in model.parameters())
        print(f"  Model: {n_params:,} parameters (k={K_ENSEMBLE})")

    model.plr.set_bins(tr_num.numpy())
    criterion = AsymmetricLoss(ASL_GAMMA_NEG, ASL_GAMMA_POS, ASL_CLIP)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best_auc, patience_counter = 0, 0
    top_states = []

    for epoch in range(EPOCHS):
        t1 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_preds, val_targets = evaluate_model(model, val_loader, device)
        macro_auc, _ = compute_macro_auc(val_targets, val_preds, target_cols)
        scheduler.step(macro_auc)

        improved = ""
        if macro_auc > best_auc:
            best_auc = macro_auc
            patience_counter = 0
            improved = " *"
        else:
            patience_counter += 1

        state_copy = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        top_states.append((macro_auc, state_copy))
        top_states.sort(key=lambda x: x[0], reverse=True)
        if len(top_states) > TOP_K_SWA:
            top_states.pop()

        print(f"    Ep {epoch+1:2d}/{EPOCHS}  loss={train_loss:.4f}  "
              f"val_auc={macro_auc:.4f}  lr={optimizer.param_groups[0]['lr']:.1e}  "
              f"{time.time()-t1:.1f}s{improved}", flush=True)

        if patience_counter >= PATIENCE:
            print(f"    Early stop at epoch {epoch+1}")
            break

    # SWA: average top-K checkpoints
    avg_state = {}
    for key in top_states[0][1]:
        avg_state[key] = sum(s[key] for _, s in top_states) / len(top_states)
    model.load_state_dict(avg_state)
    model.to(device)

    val_preds, _ = evaluate_model(model, val_loader, device)
    swa_auc, _ = compute_macro_auc(val_targets, val_preds, target_cols)
    test_preds = predict_model(model, test_loader, device)

    print(f"  Fold {fold_idx+1} best={best_auc:.4f}, SWA={swa_auc:.4f}", flush=True)
    return val_preds, test_preds, swa_auc


# ══════════════════════════════════════════════════════════════════
#  Main pipeline
# ══════════════════════════════════════════════════════════════════


def main_02():
    t0 = time.time()
    print("=" * 60)
    print(f"Step 2: Train NN (TabM + PLR + ASL + Mixup) on {DEVICE}")
    print("=" * 60)

    # 1. Load features
    print("\n[1/5] Loading features...")
    with open(FEATURES_DIR / "meta.json") as f:
        meta = json.load(f)
    cat_cols = meta["cat_cols"]
    num_cols_all = meta["num_cols"]
    target_cols = meta["target_cols"]

    train_feat = pl.read_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat = pl.read_parquet(FEATURES_DIR / "test_features.parquet")
    train_tgt = pl.read_parquet(FEATURES_DIR / "targets.parquet")
    print(f"  Features: {len(cat_cols)} cat, {len(num_cols_all)} num")

    # Cat cardinalities
    cat_cardinalities = []
    for col in cat_cols:
        combined = pl.concat([train_feat[col], test_feat[col]])
        cat_cardinalities.append(int(combined.drop_nulls().n_unique()))

    # 2. DAE pretraining
    print(f"\n[2/5] DAE Pretraining ({DAE_EPOCHS} epochs)...", flush=True)
    ENGINEERED_PREFIXES = ("is_null_", "null_count_", "null_pca_", "freq_",
                           "diff_", "ratio_", "interact_", "row_", "main_row_", "extra_row_")
    dae_input_cols = [c for c in num_cols_all if not any(c.startswith(p) for p in ENGINEERED_PREFIXES)]
    print(f"  DAE input: {len(dae_input_cols)} raw features")

    train_num_raw = train_feat.select(dae_input_cols).to_numpy()
    test_num_raw = test_feat.select(dae_input_cols).to_numpy()
    train_null_mask = np.isnan(train_num_raw).astype(np.uint8)
    test_null_mask = np.isnan(test_num_raw).astype(np.uint8)
    all_null_mask = np.vstack([train_null_mask, test_null_mask])
    del train_null_mask, test_null_mask

    all_num_data = np.vstack([
        np.nan_to_num(train_num_raw, nan=0.0).astype(np.float32),
        np.nan_to_num(test_num_raw, nan=0.0).astype(np.float32),
    ])
    del train_num_raw, test_num_raw; gc.collect()

    dae_mean = all_num_data.mean(axis=0)
    dae_std = all_num_data.std(axis=0)
    dae_std[dae_std < 1e-8] = 1.0
    all_num_data = (all_num_data - dae_mean) / dae_std

    dae_model = train_dae(all_num_data, all_null_mask, DEVICE)

    print("  Extracting DAE embeddings...")
    all_embeddings = extract_dae_embeddings(dae_model, all_num_data, DEVICE)
    n_train = train_feat.height
    train_embeddings = all_embeddings[:n_train]
    test_embeddings = all_embeddings[n_train:]
    print(f"  DAE embeddings: {all_embeddings.shape[1]} dims")

    del all_num_data, all_null_mask, dae_model, all_embeddings; gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    # Add DAE embeddings
    dae_cols = [f"dae_{i}" for i in range(DAE_BOTTLENECK_DIM)]
    train_feat = train_feat.hstack(pl.DataFrame(train_embeddings, schema=dae_cols))
    test_feat = test_feat.hstack(pl.DataFrame(test_embeddings, schema=dae_cols))
    del train_embeddings, test_embeddings

    all_num_cols = [c for c in train_feat.columns if c != "customer_id" and c not in cat_cols]
    print(f"  Total numerical features (with DAE): {len(all_num_cols)}")

    # 3. Convert test
    print("\n[3/5] Converting test to tensors...")
    test_cat, test_num = to_tensors(test_feat, cat_cols, all_num_cols, cat_cardinalities)

    # 4. K-Fold CV
    print(f"\n[4/5] Training {N_FOLDS}-Fold CV...", flush=True)
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    n_train = train_feat.height
    n_test = test_feat.height
    n_targets = len(target_cols)
    oof_preds = np.zeros((n_train, n_targets))
    test_preds_sum = np.zeros((n_test, n_targets))
    fold_aucs = []

    # Check for fold checkpoints
    resume_fold = 0
    for fi in range(N_FOLDS):
        ckpt_path = CHECKPOINT_DIR / f"fold_{fi}.npz"
        if ckpt_path.exists():
            ckpt = np.load(ckpt_path)
            oof_preds[ckpt["val_idx"]] = ckpt["val_preds"]
            test_preds_sum += ckpt["test_preds"]
            fold_aucs.append(float(ckpt["fold_auc"]))
            print(f"  Loaded fold {fi+1} (AUC={float(ckpt['fold_auc']):.4f})", flush=True)
            resume_fold = fi + 1
        else:
            break

    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(np.arange(n_train))):
        if fold_idx < resume_fold:
            continue
        print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ──", flush=True)

        tr_feat = train_feat[tr_idx.tolist()]
        val_feat_fold = train_feat[val_idx.tolist()]
        tr_tgt = train_tgt[tr_idx.tolist()]
        val_tgt = train_tgt[val_idx.tolist()]

        tr_cat, tr_num, tr_y = to_tensors(tr_feat, cat_cols, all_num_cols, cat_cardinalities, tr_tgt, target_cols)
        val_cat, val_num, val_y = to_tensors(val_feat_fold, cat_cols, all_num_cols, cat_cardinalities, val_tgt, target_cols)

        tr_num, val_num, test_num_normed = quantile_normalize(tr_num, val_num, test_num.clone())
        del tr_feat, val_feat_fold, tr_tgt, val_tgt; gc.collect()

        val_preds, test_preds, fold_auc = train_one_fold(
            fold_idx, tr_cat, tr_num, tr_y, val_cat, val_num, val_y,
            test_cat, test_num_normed, cat_cardinalities, len(all_num_cols),
            target_cols, DEVICE,
        )

        oof_preds[val_idx] = val_preds
        test_preds_sum += test_preds
        fold_aucs.append(fold_auc)

        np.savez(CHECKPOINT_DIR / f"fold_{fold_idx}.npz",
                 val_idx=val_idx, val_preds=val_preds,
                 test_preds=test_preds, fold_auc=fold_auc)

        del tr_cat, tr_num, tr_y, val_cat, val_num, val_y, test_num_normed; gc.collect()
        if DEVICE.type == "mps":
            torch.mps.empty_cache()

    # 5. Evaluation
    print(f"\n[5/5] Evaluation...", flush=True)
    oof_y = train_tgt.select(target_cols).to_numpy().astype(np.float32)
    oof_auc, _ = compute_macro_auc(oof_y, oof_preds, target_cols)
    print(f"  Per-fold AUC: {['%.4f' % a for a in fold_aucs]}")
    print(f"  OOF Macro ROC-AUC: {oof_auc:.4f}")

    # Save submission
    test_preds_avg = test_preds_sum / N_FOLDS
    
    sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
    predict_cols = [c.replace("target_", "predict_") for c in target_cols]
    submit = pl.DataFrame({"customer_id": test_feat["customer_id"]}).hstack(
        pl.DataFrame(test_preds_avg.astype(np.float64), schema=predict_cols)
    )
    verify_submission(submit, sample)
    SUB_DIR.mkdir(exist_ok=True)
    submit.write_parquet("submissions/nn.parquet")
    print(f"  Saved: submissions/nn.parquet")

    print(f"\nDone in {(time.time()-t0)/60:.1f} min. OOF={oof_auc:.4f}")




In [ ]:
# Run NN training (quality preset, <=12h H100 target)
seed_everything(SEED)

# High-quality preset
EPOCHS = 72
PATIENCE = 14
TOP_K_SWA = 8
K_ENSEMBLE = 24
MIXUP_ALPHA = 0.25
DAE_EPOCHS = 28
DAE_BOTTLENECK_DIM = 320

print('NN preset:', {
    'EPOCHS': EPOCHS, 'PATIENCE': PATIENCE, 'TOP_K_SWA': TOP_K_SWA,
    'K_ENSEMBLE': K_ENSEMBLE, 'DAE_EPOCHS': DAE_EPOCHS,
    'DAE_BOTTLENECK_DIM': DAE_BOTTLENECK_DIM
})
main_02()


## 3) LightGBM training code


In [ ]:
"""Step 3: Train LightGBM (Optuna-tuned params, 5-fold).

Loads features from features/, trains 41 per-target binary classifiers.

Output: checkpoints_lgbm/lgbm_predictions.npz (oof_preds, test_preds)

Runtime: ~20-30 minutes.
"""

import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold



FEATURES_DIR = FEATURES_DIR
CHECKPOINT_DIR = CKPT_LGBM

# Optuna-tuned params (L7, 30 trials)
LGBM_PARAMS = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.050216,
    num_leaves=34,
    max_depth=10,
    min_child_samples=102,
    n_estimators=1500,
    subsample=0.521715,
    colsample_bytree=0.205092,
    reg_alpha=8.146025,
    reg_lambda=7.761833,
    min_split_gain=0.424512,
    subsample_freq=2,
    random_state=SEED,
    verbose=-1,
    n_jobs=-1,
    device_type="gpu",
    gpu_use_dp=False,
)
EARLY_STOPPING_ROUNDS = 100


def main_03():
    t0 = time.time()
    print("=" * 60)
    print("Step 3: Train LightGBM (5-fold × 41 targets)")
    print("=" * 60)

    # 1. Load features
    print("\n[1/4] Loading features...")
    with open(FEATURES_DIR / "meta.json") as f:
        meta = json.load(f)
    feature_cols = meta["feature_names"]
    cat_feature_names = meta["cat_cols"]
    target_cols = meta["target_cols"]
    cat_indices = [feature_cols.index(c) for c in cat_feature_names]

    train_feat = pl.read_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat = pl.read_parquet(FEATURES_DIR / "test_features.parquet")
    train_tgt = pl.read_parquet(FEATURES_DIR / "targets.parquet")

    X_train = train_feat.drop("customer_id").to_numpy().astype(np.float32)
    X_test = test_feat.drop("customer_id").to_numpy().astype(np.float32)
    y_train = train_tgt.select(target_cols).to_numpy().astype(np.float32)
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
    print(f"  Features: {len(cat_feature_names)} cat, {len(feature_cols) - len(cat_feature_names)} num")

    # 2. Check cache
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = CHECKPOINT_DIR / "lgbm_predictions.npz"
    if cache_file.exists():
        print(f"\n  Predictions exist at {cache_file}! Delete to retrain.")
        return

    # 3. Train
    n_train, n_test = X_train.shape[0], X_test.shape[0]
    n_targets = len(target_cols)
    oof_preds = np.zeros((n_train, n_targets))
    test_preds_sum = np.zeros((n_test, n_targets))
    fold_aucs = []

    kf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    print(f"\n[2/4] Training {N_FOLDS}-Fold × {n_targets} targets...", flush=True)

    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(np.arange(n_train), y_train)):
        t_fold = time.time()
        print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} "
              f"(train={len(tr_idx):,}, val={len(val_idx):,}) ──", flush=True)

        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        fold_test_preds = np.zeros((n_test, n_targets))

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            for i, col in enumerate(target_cols):
                model = lgb.LGBMClassifier(**LGBM_PARAMS)
                model.fit(
                    X_tr, y_tr[:, i],
                    eval_set=[(X_val, y_val[:, i])],
                    eval_metric="auc",
                    callbacks=[
                        lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                        lgb.log_evaluation(period=0),
                    ],
                    categorical_feature=cat_indices,
                )
                oof_preds[val_idx, i] = model.predict_proba(X_val)[:, 1]
                fold_test_preds[:, i] = model.predict_proba(X_test)[:, 1]
                del model
                if (i + 1) % 10 == 0 or i == n_targets - 1:
                    print(f"    {i+1}/{n_targets} targets done", flush=True)

        fold_auc, _ = compute_macro_auc(y_val, oof_preds[val_idx], target_cols)
        test_preds_sum += fold_test_preds
        fold_aucs.append(fold_auc)
        del X_tr, X_val, y_tr, y_val; gc.collect()
        print(f"  Fold {fold_idx+1} AUC={fold_auc:.4f} ({(time.time()-t_fold)/60:.1f} min)", flush=True)

    test_preds_avg = test_preds_sum / N_FOLDS

    # 4. Results
    oof_auc, _ = compute_macro_auc(y_train, oof_preds, target_cols)
    print(f"\n[3/4] Results:")
    print(f"  Per-fold AUC: {['%.4f' % a for a in fold_aucs]}")
    print(f"  OOF Macro ROC-AUC: {oof_auc:.4f}")

    # Save predictions
    np.savez(cache_file, oof_preds=oof_preds, test_preds=test_preds_avg,
             fold_aucs=np.array(fold_aucs))
    print(f"  Saved: {cache_file}")

    # Save submission
    print("\n[4/4] Saving submission...")
    
    sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
    predict_cols = [c.replace("target_", "predict_") for c in target_cols]
    submit = pl.DataFrame({"customer_id": test_feat["customer_id"]}).hstack(
        pl.DataFrame(test_preds_avg.astype(np.float64), schema=predict_cols)
    )
    verify_submission(submit, sample)
    SUB_DIR.mkdir(exist_ok=True)
    submit.write_parquet("submissions/lgbm.parquet")
    print(f"  Saved: submissions/lgbm.parquet")

    print(f"\nDone in {(time.time()-t0)/60:.1f} min. OOF={oof_auc:.4f}")




In [ ]:
# Run LGBM training (high-quality GPU preset, <=12h H100 target)
seed_everything(SEED)

LGBM_PARAMS.update({
    'n_estimators': 4200,
    'learning_rate': 0.028,
    'num_leaves': 64,
    'max_depth': -1,
    'min_child_samples': 64,
    'subsample': 0.75,
    'colsample_bytree': 0.45,
    'reg_alpha': 5.0,
    'reg_lambda': 9.0,
})
EARLY_STOPPING_ROUNDS = 350

print('LGBM preset:', {k:LGBM_PARAMS[k] for k in ['n_estimators','learning_rate','num_leaves','colsample_bytree','device_type']})
print('EARLY_STOPPING_ROUNDS=', EARLY_STOPPING_ROUNDS)
main_03()


## 4) PyBoost training code


In [ ]:
"""Step 4: Train PyBoost (SketchBoost, requires NVIDIA GPU with CUDA).

Loads features from features/, trains SketchBoost multi-output model.

Output: checkpoints_pyboost/pyboost_predictions.npz (oof_preds, test_preds)

Runtime: ~1-2 hours on T4 GPU.
"""

import gc
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold



FEATURES_DIR = FEATURES_DIR
CHECKPOINT_DIR = CKPT_PB

# Optuna-tuned params (25 trials on fold 0)
PARAMS = dict(
    ntrees=5000,
    lr=0.0566,
    max_depth=7,
    min_data_in_leaf=88,
    lambda_l2=2.076,
    subsample=0.88,
    colsample=0.76,
    max_bin=64,
    es=200,
    verbose=100,
    gd_steps=1,
    use_hess=False,
)


def main_04():
    t0 = time.time()
    print("=" * 60)
    print("Step 4: Train PyBoost (SketchBoost)")
    print("=" * 60)

    # Check CUDA
    try:
        import torch
        if not torch.cuda.is_available():
            print("\nERROR: PyBoost requires NVIDIA GPU with CUDA.")
            print("  This script uses cupy + py-boost which only work on CUDA GPUs.")
            print("  Skipping PyBoost training.")
            return
    except ImportError:
        print("\nERROR: torch not found. Install PyTorch with CUDA support.")
        return

    try:
        import cupy as cp
        from py_boost import SketchBoost
    except ImportError:
        print("\nERROR: cupy and/or py-boost not installed.")
        print("  pip install py-boost cupy-cuda12x")
        return

    # 1. Load features
    print("\n[1/4] Loading features...")
    with open(FEATURES_DIR / "meta.json") as f:
        meta = json.load(f)
    target_cols = meta["target_cols"]

    train_feat = pl.read_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat = pl.read_parquet(FEATURES_DIR / "test_features.parquet")
    train_tgt = pl.read_parquet(FEATURES_DIR / "targets.parquet")

    X_train = train_feat.drop("customer_id").to_numpy().astype(np.float32)
    X_test = test_feat.drop("customer_id").to_numpy().astype(np.float32)
    y = train_tgt.select(target_cols).to_numpy().astype(np.float32)
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")

    # 2. Check cache
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = CHECKPOINT_DIR / "pyboost_predictions.npz"
    if cache_file.exists():
        print(f"\n  Predictions exist at {cache_file}! Delete to retrain.")
        return

    # 3. Train
    print(f"\n[2/4] Training SketchBoost {N_FOLDS}-fold...")
    mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_preds = np.zeros_like(y, dtype=np.float64)
    test_preds = np.zeros((X_test.shape[0], len(target_cols)), dtype=np.float64)
    fold_scores = []

    for fold_idx, (tr_idx, val_idx) in enumerate(mskf.split(X_train, y)):
        t_fold = time.time()
        print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ──", flush=True)

        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model = SketchBoost('bce', **PARAMS)
        model.fit(X_tr, y_tr, eval_sets=[{'X': X_val, 'y': y_val}])

        val_raw = model.predict(X_val)
        test_raw = model.predict(X_test)
        val_prob = 1.0 / (1.0 + np.exp(-val_raw))
        test_prob = 1.0 / (1.0 + np.exp(-test_raw))

        oof_preds[val_idx] = val_prob
        test_preds += test_prob / N_FOLDS

        fold_auc = roc_auc_score(y_val, val_prob, average='macro')
        fold_scores.append(fold_auc)
        print(f"  Fold {fold_idx+1} AUC={fold_auc:.5f} ({(time.time()-t_fold)/60:.1f} min)", flush=True)

        del model, X_tr, X_val, y_tr, y_val, val_raw, test_raw, val_prob, test_prob
        cp.get_default_memory_pool().free_all_blocks(); gc.collect()

    # 4. Results
    macro_auc = float(np.mean([
        roc_auc_score(y[:, i], oof_preds[:, i]) for i in range(len(target_cols))
    ]))
    print(f"\n[3/4] Results:")
    print(f"  Fold AUCs: {['%.5f' % s for s in fold_scores]}")
    print(f"  OOF Macro AUC: {macro_auc:.5f}")

    # Save
    np.savez_compressed(cache_file,
                        oof_preds=oof_preds, test_preds=test_preds,
                        fold_scores=fold_scores, macro_auc=macro_auc)
    print(f"  Saved: {cache_file}")

    # Save submission
    print("\n[4/4] Saving submission...")
    
    sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
    predict_cols = [c.replace("target_", "predict_") for c in target_cols]
    submit = pl.DataFrame({"customer_id": test_feat["customer_id"]}).hstack(
        pl.DataFrame(test_preds.astype(np.float64), schema=predict_cols)
    )
    verify_submission(submit, sample)
    SUB_DIR.mkdir(exist_ok=True)
    submit.write_parquet("submissions/pyboost.parquet")
    print(f"  Saved: submissions/pyboost.parquet")

    print(f"\nDone in {(time.time()-t0)/60:.1f} min. OOF={macro_auc:.5f}")




In [ ]:
# Run PyBoost training (high-quality preset, <=12h H100 target)
seed_everything(SEED)

PARAMS.update({
    'ntrees': 7600,
    'lr': 0.042,
    'max_depth': 8,
    'min_data_in_leaf': 64,
    'lambda_l2': 2.6,
    'subsample': 0.92,
    'colsample': 0.82,
    'es': 320,
    'max_bin': 96,
    'gd_steps': 2,
})
print('PyBoost preset:', PARAMS)
main_04()


## 5) LGBM-meta (cross-target) training code


In [ ]:
"""Step 5: Train LGBM with cross-target meta-features (L8).

Uses OOF predictions from NN, LGBM, and PyBoost as additional features.
For each target_i: base features + 120 meta-features (40 from each model,
excluding the own target to prevent leakage).

Output: checkpoints_lgbm_meta/lgbm_predictions.npz

Runtime: ~30-40 minutes.
"""

import gc
import json
import os
import sys
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold



FEATURES_DIR = FEATURES_DIR
CHECKPOINT_DIR = CKPT_LGBM_META

# Same Optuna-tuned params as L7
LGBM_PARAMS = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.050216,
    num_leaves=34,
    max_depth=10,
    min_child_samples=102,
    n_estimators=1500,
    subsample=0.521715,
    colsample_bytree=0.205092,
    reg_alpha=8.146025,
    reg_lambda=7.761833,
    min_split_gain=0.424512,
    subsample_freq=2,
    random_state=SEED,
    verbose=-1,
    n_jobs=-1,
    device_type="gpu",
    gpu_use_dp=False,
)
EARLY_STOPPING_ROUNDS = 100


def load_model_predictions():
    """Load OOF and test predictions from all 3 models."""
    print("\n  Loading OOF predictions from 3 models...")

    # NN
    nn_dir = CKPT_NN
    n_train_check = np.load(nn_dir / "fold_0.npz")["val_preds"].shape[1]
    oof_parts, test_parts = {}, []
    for fi in range(N_FOLDS):
        d = np.load(nn_dir / f"fold_{fi}.npz")
        for idx, pred in zip(d["val_idx"], d["val_preds"]):
            oof_parts[int(idx)] = pred
        test_parts.append(d["test_preds"])
    n_train = max(oof_parts.keys()) + 1
    nn_oof = np.zeros((n_train, n_train_check), dtype=np.float32)
    for idx, pred in oof_parts.items():
        nn_oof[idx] = pred
    nn_test = np.mean(test_parts, axis=0).astype(np.float32)
    print(f"    NN: OOF {nn_oof.shape}, test {nn_test.shape}")

    # LGBM
    d = np.load("checkpoints_lgbm/lgbm_predictions.npz")
    lgbm_oof = d["oof_preds"].astype(np.float32)
    lgbm_test = d["test_preds"].astype(np.float32)
    print(f"    LGBM: OOF {lgbm_oof.shape}, test {lgbm_test.shape}")

    # PyBoost
    d = np.load("checkpoints_pyboost/pyboost_predictions.npz")
    pb_oof = d["oof_preds"].astype(np.float32)
    pb_test = d["test_preds"].astype(np.float32)
    print(f"    PyBoost: OOF {pb_oof.shape}, test {pb_test.shape}")

    meta_train = np.hstack([lgbm_oof, nn_oof, pb_oof])
    meta_test = np.hstack([lgbm_test, nn_test, pb_test])
    print(f"    Combined meta: {meta_train.shape}")
    return meta_train, meta_test


def main_05():
    t0 = time.time()
    print("=" * 60)
    print("Step 5: LGBM with cross-target meta-features (L8)")
    print("=" * 60)

    # 1. Load base features
    print("\n[1/4] Loading features...")
    with open(FEATURES_DIR / "meta.json") as f:
        meta = json.load(f)
    feature_cols = meta["feature_names"]
    cat_feature_names = meta["cat_cols"]
    target_cols = meta["target_cols"]
    cat_indices = [feature_cols.index(c) for c in cat_feature_names]

    train_feat = pl.read_parquet(FEATURES_DIR / "train_features.parquet")
    test_feat = pl.read_parquet(FEATURES_DIR / "test_features.parquet")
    train_tgt = pl.read_parquet(FEATURES_DIR / "targets.parquet")

    X_train_base = train_feat.drop("customer_id").to_numpy().astype(np.float32)
    X_test_base = test_feat.drop("customer_id").to_numpy().astype(np.float32)
    y_train = train_tgt.select(target_cols).to_numpy().astype(np.float32)

    n_targets = len(target_cols)
    n_base = len(feature_cols)

    # 2. Load meta predictions
    meta_train, meta_test = load_model_predictions()

    # Combine
    X_train_full = np.hstack([X_train_base, meta_train])
    X_test_full = np.hstack([X_test_base, meta_test])
    del X_train_base, X_test_base, meta_train, meta_test; gc.collect()

    n_total = X_train_full.shape[1]
    print(f"\n  Features: {n_base} base + {n_total - n_base} meta = {n_total} total")

    # 3. Check cache
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    cache_file = CHECKPOINT_DIR / "lgbm_predictions.npz"
    if cache_file.exists():
        print(f"\n  Predictions exist at {cache_file}! Delete to retrain.")
        return

    # Precompute own-target column indices to mask
    # For target_i: mask columns [n_base+i, n_base+41+i, n_base+82+i]
    own_cols = np.array([
        [n_base + i, n_base + n_targets + i, n_base + 2 * n_targets + i]
        for i in range(n_targets)
    ])

    # 4. Train
    print(f"\n[2/4] Training {N_FOLDS}-Fold × {n_targets} targets (with meta)...", flush=True)
    kf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    n_train, n_test = X_train_full.shape[0], X_test_full.shape[0]

    oof_preds = np.zeros((n_train, n_targets))
    test_preds_sum = np.zeros((n_test, n_targets))
    fold_aucs = []

    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(np.arange(n_train), y_train)):
        t_fold = time.time()
        print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ──", flush=True)

        X_tr = X_train_full[tr_idx]
        X_val = X_train_full[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        fold_test_preds = np.zeros((n_test, n_targets))

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            for i, col in enumerate(target_cols):
                cols_to_mask = own_cols[i]

                # Mask own-target meta-features to NaN
                saved_tr = X_tr[:, cols_to_mask].copy()
                saved_val = X_val[:, cols_to_mask].copy()
                saved_test = X_test_full[:, cols_to_mask].copy()
                X_tr[:, cols_to_mask] = np.nan
                X_val[:, cols_to_mask] = np.nan
                X_test_full[:, cols_to_mask] = np.nan

                model = lgb.LGBMClassifier(**LGBM_PARAMS)
                model.fit(
                    X_tr, y_tr[:, i],
                    eval_set=[(X_val, y_val[:, i])],
                    eval_metric="auc",
                    callbacks=[
                        lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                        lgb.log_evaluation(period=0),
                    ],
                    categorical_feature=cat_indices,
                )
                oof_preds[val_idx, i] = model.predict_proba(X_val)[:, 1]
                fold_test_preds[:, i] = model.predict_proba(X_test_full)[:, 1]

                # Restore
                X_tr[:, cols_to_mask] = saved_tr
                X_val[:, cols_to_mask] = saved_val
                X_test_full[:, cols_to_mask] = saved_test
                del model, saved_tr, saved_val, saved_test

                if (i + 1) % 10 == 0 or i == n_targets - 1:
                    print(f"    {i+1}/{n_targets} targets done", flush=True)

        fold_auc, _ = compute_macro_auc(y_val, oof_preds[val_idx], target_cols)
        test_preds_sum += fold_test_preds
        fold_aucs.append(fold_auc)
        del X_tr, X_val, y_tr, y_val; gc.collect()
        print(f"  Fold {fold_idx+1} AUC={fold_auc:.4f} ({(time.time()-t_fold)/60:.1f} min)", flush=True)

    test_preds_avg = test_preds_sum / N_FOLDS

    # Results
    oof_auc, _ = compute_macro_auc(y_train, oof_preds, target_cols)
    print(f"\n[3/4] Results:")
    print(f"  Per-fold AUC: {['%.4f' % a for a in fold_aucs]}")
    print(f"  OOF Macro ROC-AUC: {oof_auc:.4f}")

    np.savez(cache_file, oof_preds=oof_preds, test_preds=test_preds_avg,
             fold_aucs=np.array(fold_aucs))
    print(f"  Saved: {cache_file}")

    # Save submission
    print("\n[4/4] Saving submission...")
    
    sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
    predict_cols = [c.replace("target_", "predict_") for c in target_cols]
    submit = pl.DataFrame({"customer_id": test_feat["customer_id"]}).hstack(
        pl.DataFrame(test_preds_avg.astype(np.float64), schema=predict_cols)
    )
    verify_submission(submit, sample)
    SUB_DIR.mkdir(exist_ok=True)
    submit.write_parquet("submissions/lgbm_meta.parquet")
    print(f"  Saved: submissions/lgbm_meta.parquet")

    print(f"\nDone in {(time.time()-t0)/60:.1f} min. OOF={oof_auc:.4f}")




In [ ]:
# Run LGBM-meta training (high-quality GPU preset, <=12h H100 target)
seed_everything(SEED)

LGBM_PARAMS.update({
    'n_estimators': 3200,
    'learning_rate': 0.03,
    'num_leaves': 48,
    'max_depth': -1,
    'min_child_samples': 72,
    'subsample': 0.78,
    'colsample_bytree': 0.38,
    'reg_alpha': 5.0,
    'reg_lambda': 9.0,
})
EARLY_STOPPING_ROUNDS = 260

print('LGBM-meta preset:', {k:LGBM_PARAMS[k] for k in ['n_estimators','learning_rate','num_leaves','colsample_bytree','device_type']})
print('EARLY_STOPPING_ROUNDS=', EARLY_STOPPING_ROUNDS)
main_05()


In [ ]:
# Checkpoint summary
from pathlib import Path
for p in [CKPT_NN, CKPT_LGBM, CKPT_PB, CKPT_LGBM_META, BLEND_ART, SUB_DIR]:
    files = list(Path(p).glob('*'))
    print(f"{p}: {len(files)} files")
    for f in files[:5]:
        print('  -', f.name)
